# Week 4: Conservation and the profile

| | |
|---|---|
| **Due** | start of the Week 5 meeting |
| **Prerequisites** | Weeks 1–3 |
| **Produces** | **`week04_dhfr_msa.fasta` — you will reuse this in Week 7. Do not lose it.** |

## Learning objectives

1. Build and interpret a multiple sequence alignment.
2. Quantify per-column conservation and relate it to protein function.
3. Explain how database redundancy distorts apparent conservation.
4. Build a position-specific scoring matrix and say why it finds what a single sequence cannot.
5. Map alignment columns back to residue numbers without off-by-one errors.

## Where this sits

You can align two sequences. Aligning many at once is not simply more of the same: it lets you see **which positions a protein family cannot afford to change**, which is the closest thing sequence analysis offers to a readout of function.

It also matters for a reason that will not be obvious until later in Week 7 when we talk about AlphaFold. AlphaFold's confidence in a predicted structure is largely a function of how much evolutionary signal was available in the alignment for that protein. 

In [1]:
#@title Setup — run this first
%pip install -q "biopython>=1.85" "pandas>=2.0" "matplotlib>=3.7" "requests>=2.31"
print("setup complete")

Note: you may need to restart the kernel to use updated packages.
setup complete


In [2]:
import pathlib, requests
FIXTURES = pathlib.Path("fixtures")
COURSE_DATA_URL = None   

def course_file(name: str) -> pathlib.Path:
    local = FIXTURES / name
    if local.exists():
        return local
    if COURSE_DATA_URL:
        FIXTURES.mkdir(exist_ok=True)
        r = requests.get(f"{COURSE_DATA_URL}/{name}", timeout=30); r.raise_for_status()
        local.write_bytes(r.content); return local
    raise FileNotFoundError(f"{name} not found. Put fixtures/ beside this notebook.")

print("fixtures:", FIXTURES.resolve())

fixtures: C:\Users\laura\OneDrive - James Madison University\Documents\ISAT\454\compbio\fixtures


---
## Part 1: Build the alignment

You have thirteen reviewed DHFR sequences — twelve bacterial plus **human DHFR (`P00374`)**. Align them with **Clustal Omega** at
https://www.ebi.ac.uk/jdispatcher/msa/clustalo — upload the FASTA, submit, and download the result in FASTA (aligned) format as `week04_dhfr_msa.fasta`.

A fallback alignment has also been uploaded to the fixtures folder in case the Clustal Omega is unavailable. It was produced by a simple center-star method, **not** by Clustal Omega (i.e., it won't be exactly the same), and it is a fallback rather than a reference answer. If you do need to use it, please indicate as such.

In [ ]:
from Bio import AlignIO, SeqIO
import pathlib

target = pathlib.Path("week04_dhfr_msa.fasta")
if target.exists():
    aln = AlignIO.read(target, "fasta")
    print("using your Clustal Omega alignment")
else:
    aln = AlignIO.read(course_file("dhfr_homologs_aligned_FALLBACK.fasta"), "fasta")
    print("using the FALLBACK center-star alignment (note this in your submission)")

print(f"{len(aln)} sequences x {aln.get_alignment_length()} columns")
for rec in aln:
    print(" ", rec.id.split("|")[1], rec.id.split("|")[2])

---
## Part 2: Conservation, and a trap

The obvious way to measure conservation at a column is Shannon entropy: low entropy means everyone agrees, high entropy means the position varies freely.

Compute it. Then look hard at your sequence list before believing the numbers. Think about how you might evaluate AI output here - don't blindly trust it.

In [ ]:
import math
import numpy as np

def column_entropy(column: str):
    """Shannon entropy of an alignment column, in bits. Gaps ignored."""
    residues = [c for c in column if c != "-"]
    if not residues:
        return None
    n = len(residues)
    counts = {}
    for c in residues:
        counts[c] = counts.get(c, 0) + 1
    return -sum((v / n) * math.log2(v / n) for v in counts.values())

seqs = [str(r.seq) for r in aln]
accs = [r.id.split("|")[1] for r in aln]
L = aln.get_alignment_length()

entropies = [column_entropy("".join(s[j] for s in seqs)) for j in range(L)]
valid = [e for e in entropies if e is not None]
print(f"mean entropy across {len(valid)} columns: {np.mean(valid):.3f} bits")
print(f"perfectly conserved columns (entropy 0): {sum(1 for e in valid if e == 0)}")

### The trap

Look at your accession list. There are three groups of identical sequences in it. `P0ABQ4`, `P0ABQ5`, and `P0ABQ6` are the same sequence — *E. coli* K-12, *E. coli* CFT073, and *Shigella flexneri*. `P0A016` and `P0A017` are two *S. aureus* strains. `P9WNX1` and `P0A547` are *M. tuberculosis* and *M. bovis*.

If we remove three duplicates from twelve rows, that leaves **eight** distinct sequences. However, every column is being told the *E. coli* answer three times, and the *S. aureus* and *Mycobacterium* answers twice each.

Measure what that does.

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: the same entropy calculation on the deduplicated set, and a comparison of the two
# Example: report mean entropy for all 12 vs the 8 unique, and count how many columns change
# ---- END ----


---
## Part 3: Where are the conserved positions?

A conservation number is only useful once you can say *which residue* it refers to. That means mapping alignment columns back to positions in a specific sequence. Be careful! This is where off-by-one errors breed, especially wen comparing sequences of different lengths.

Build the map and test it.

In [ ]:
ECOLI_ROW = accs.index("P0ABQ4")
ecoli_aln = seqs[ECOLI_ROW]

col_to_res, res_to_col = {}, {}
res_no = 0
for col, ch in enumerate(ecoli_aln):
    if ch != "-":
        res_no += 1
        col_to_res[col] = res_no
        res_to_col[res_no] = col

print(f"mapped {len(col_to_res)} columns to residues 1..{res_no}")
assert res_no == 159, f"expected 159 residues for P0ABQ4, got {res_no}"

# Spot-check against the known sequence: residue 27 of E. coli DHFR is aspartate,
# the catalytic acid of the active site.
plain = ecoli_aln.replace("-", "")
print(f"residue 27 is {plain[26]}  (expect D)")
assert plain[26] == "D"
print("mapping verified")

Asp27 is the catalytic acid in the *E. coli* DHFR active site — it protonates the substrate — and it is where trimethoprim binds. If your column-to-residue map is wrong, every conservation claim you make afterwards will point at the wrong amino acid. Unfortunately, this is a very silent error and hard to catch. 

NOTE: You will need this exact map again in Week 7. Save it!!

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: the ten most conserved columns, converted to E. coli residue numbers with their amino acids
# Example: your list should include residue 27 (D); if it does not, check your mapping
# ---- END ----


---
## Part 4: From alignment to profile

A single sequence asks the question "how similar are you to *me*?" A **profile** on the other hand asks "how well do you fit this *family*?" This is a much better question, because it helps identify which positions are negotiable.

Build a position-specific scoring matrix from your alignment: for each column, the frequency of each amino acid.

In [ ]:
import pandas as pd

AAS = "ACDEFGHIKLMNPQRSTVWY"

# Background amino-acid frequencies. 
BACKGROUND = {aa: 1 / len(AAS) for aa in AAS}

def build_pssm(sequences, pseudocount=0.1):
    """Log-odds position-specific scoring matrix.

    A frequency table says how often each residue appears in a column. That is not yet a
    scoring matrix. A PSSM scores a residue by how much MORE often it appears here than
    background chance would predict -- log2(observed / expected) -- so a column that is 90%
    glycine scores glycine highly and everything else poorly, while a column matching
    background scores near zero for everything and contributes nothing to a search.
    """
    n_cols = len(sequences[0])
    rows = []
    for j in range(n_cols):
        col = [s[j] for s in sequences if s[j] != "-"]
        total = len(col) + pseudocount * len(AAS)
        row = {}
        for aa in AAS:
            freq = (col.count(aa) + pseudocount) / total
            row[aa] = math.log2(freq / BACKGROUND[aa])
        rows.append(row)
    return pd.DataFrame(rows)

pssm = build_pssm([seqs[i] for i in keep])
print("PSSM shape:", pssm.shape)

col27 = res_to_col[27]
print(f"\nLog-odds scores at the Asp27 column (alignment column {col27}):")
print(pssm.iloc[col27].sort_values(ascending=False).head(4).to_string())

---
## Extensions

### Computing extension: entropy versus naive identity

Compare your entropy ranking against a simpler measure: the percentage of sequences carrying the most common residue at each column. Where do the two disagree, and which is more informative?

Consider a column that is 50% alanine and 50% glycine, against one that is 50% alanine and 50% of ten different residues spread evenly. Naive percent identity calls both 50%. Entropy does not.

> **Next week you will do this a third way.** A PSSM is an explicit model of a family: you can
> read every number in it and say where it came from. Next week, you will compare it against a
> protein language model, which represents the same family as a vector you cannot inspect at all,
> and ask which one you believe when they disagree.

### Biology extension: do the conserved positions mean anything?

Take your five most conserved positions and determine, from the primary literature on DHFR, whether each is catalytic, substrate- or cofactor-binding, or structural. Conservation is evidence *of constraint*, not evidence of any particular kind of constraint — a buried residue essential to folding is as conserved as a catalytic one, for entirely different reasons.

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: your chosen extension(s), clearly labelled
# Example: for the computing extension, a table of columns where the two measures disagree most
# ---- END ----

### Now use it

A profile is only worth building if it finds something a single sequence would not. Score each
of your twelve sequences against the profile, then compare that ranking to a plain pairwise
alignment against *E. coli* DHFR alone.

The profile knows which columns are negotiable and which are not. A pairwise alignment does
not — it treats every position as equally informative. When we have a whole family of sequences, we can drop the assumption that every position is equally informative.

In [ ]:
# ---- YOUR CODE HERE ----
# Expected: each sequence scored against the PSSM, and that ranking compared with plain pairwise scores against E. coli DHFR
# Example: report both rankings side by side and name one sequence whose position changes
# ---- END ----


---
## Part 5: Save the alignment!

**This file is an input to Week 7.** Save it, keep it, and do not regenerate it later with different parameters. Week 7's analysis will compare per-residue structure confidence against per-column conservation from *this* alignment.

In [ ]:
from Bio import AlignIO
AlignIO.write(aln, "week04_dhfr_msa.fasta", "fasta")
print("wrote week04_dhfr_msa.fasta")
print(f"  {len(aln)} sequences, {aln.get_alignment_length()} columns")
print("  Keep this file.")

---
## Submission checklist

- [ ] Alignment built (note whether Clustal Omega or the fallback)
- [ ] Per-column entropy computed
- [ ] Deduplication comparison run, with your reading of the size of the effect
- [ ] Column-to-residue map built and the Asp27 assertion passing
- [ ] Ten most conserved positions reported as residue numbers
- [ ] PSSM built
- [ ] `week04_dhfr_msa.fasta` saved
- [ ] At least one extension, clearly labelled
- [ ] Notebook runs top to bottom in a fresh runtime